# <b>Retrieval Augmented Generation</b>

## <b> Indexing </b>

### <b><i><u>Document Loading

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
import copy
from rich import print as rprint

In [ ]:
loader_pdf = PyPDFLoader("Introduction_to_Data_and_Data_Science.pdf")

In [ ]:
pages = loader_pdf.load()

In [ ]:
rprint(pages)

In [ ]:
page_pdf_cut = copy.deepcopy(pages)

In [ ]:
for page in page_pdf_cut:
    page.page_content = " ".join(page.page_content.split())

In [ ]:
rprint(page_pdf_cut)

In [ ]:
from langchain_community.document_loaders import Docx2txtLoader

loader_docx = Docx2txtLoader("testdoc.docx")
pages = loader_docx.load()

In [ ]:
rprint(pages)

---

### <b> <i> <u>Document Splitting

Charater Text Splitter : 
Segments text based on a separator, maximum chuck size, and chunk overlap.

In [ ]:
from langchain_text_splitters import CharacterTextSplitter
doc_length = len(pages[0].page_content)

In [ ]:
char_splitter = CharacterTextSplitter(
    separator='',
    chunk_size = 500,
    chunk_overlap = 0 # expirement with overlap (50)
)

In [ ]:
pages_char_split = char_splitter.split_documents(pages)

In [ ]:
rprint(pages_char_split)

In [ ]:
rprint(pages_char_split[10].page_content)

Markdown Header text splitter :
Based on splitting concerning headers

In [ ]:
from langchain_community.document_loaders import Docx2txtLoader
loader_docx = Docx2txtLoader("textdoc_markdown.docx")
docx_pages = loader_docx.load()

In [ ]:
rprint(docx_pages)

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=[
    ("#", "Course Title"),
    ("##", "Lecture Title")
    ])

In [ ]:
pages_md_split = markdown_splitter.split_text(docx_pages[0].page_content)

In [ ]:
rprint(pages_md_split)

other langchain text splitter
* Token Text Splitter
* Recursive Character Text Splitter


### <b><i><u>Document Embedding 

In [ ]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import CharacterTextSplitter, MarkdownHeaderTextSplitter
from langchain_openai.embeddings import OpenAIEmbeddings
import numpy as np


In [ ]:
loader = Docx2txtLoader('textdoc_markdown.docx')
pages = loader.load()

In [ ]:
# splitting doucument into chunks using the markdown header strategy
md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=[
    ("#", "Course title"),
    ("##", "Lecture title"),
])

pages_md_split =  md_splitter.split_text(pages[0].page_content)

# cleaning data (removing /n from document)
for page in range(len(pages_md_split)):
    pages_md_split[page].page_content = " ".join(pages_md_split[page].page_content.split())

# splitting document into chuck using the charscter text strategy
charater_splitter = CharacterTextSplitter(
    separator=".",
    chunk_size = 500,
    chunk_overlap = 50
)

pages = charater_splitter.split_documents(pages_md_split)

In [ ]:
rprint(len(pages))

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")

In [ ]:
vector1 = embeddings.embed_query(pages[3].page_content)
vector2 = embeddings.embed_query(pages[5].page_content)
vector3 = embeddings.embed_query(pages[18].page_content)

In [ ]:
len(vector1), len(vector2), len(vector3)

In [ ]:
np.dot(vector1, vector2), np.dot(vector1, vector3), np.dot(vector2, vector3)

In [ ]:
np.linalg.norm(vector1), np.linalg.norm(vector2), np.linalg.norm(vector3)

### <b><i><u> Documents storing

In [ ]:
# Workaround: force pure-Python protobuf implementation (no kernel restart needed)
# import os
# os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

# If the above doesn't work, pin protobuf and restart the kernel:
# %pip install -U "protobuf==3.20.3"
import chromadb
from langchain_chroma import Chroma

In [ ]:
vectorstore = Chroma.from_documents(
    documents=pages,
    embedding=embeddings,
    persist_directory="./vectorstore_db"
)

In [ ]:
vectorstore_from_dir = Chroma(
    persist_directory="./vectorstore_db",
    embedding_function=embeddings
)

Inspecting and managing contents on vectorstore

In [ ]:
vectorstore_from_dir.get()

In [ ]:
vectorstore_from_dir.get(ids="687e2a8b-be79-4693-9925-e4c73d7891a3", include = ["embeddings"])

adding new documents to vectorstore

In [ ]:
# adding new documents to vectorstore
from langchain_core.documents import Document
add_doc = Document(page_content='Alright! So… Let’s discuss the not-so-obvious differences between the terms analysis and analytics. Due to the similarity of the words, some people believe they share the same meaning, and thus use them interchangeably. Technically, this isn’t correct. There is, in fact, a distinct difference between the two. And the reason for one often being used instead of the other is the lack of a transparent understanding of both. So, let’s clear this up, shall we? First, we will start with analysis', 
                          metadata={'Course Title': 'Introduction to Data and Data Science', 
                                    'Lecture Title': 'Analysis vs Analytics'})

In [ ]:
vectorstore_from_dir.add_documents([add_doc])

In [ ]:
vectorstore_from_dir.get(ids= "863c6520-c00d-4e73-b4d6-d94a9de52d76", include=["embeddings"])

updating vectorstore

In [ ]:
update_doc = Document(
        page_content='Data is defined as information, especially facts or numbers, collected for examination and consideration. Data can be structured or unstructured. Structured data is organized in rows and columns like a spreadsheet. Unstructured data has no predefined format, like emails, images, and social media posts.',
        metadata={'Course Title': 'Introduction to Data and Data Science',
                  'Lecture Title': 'What is Data'}
    )

In [ ]:
vectorstore_from_dir.update_document(document_id="863c6520-c00d-4e73-b4d6-d94a9de52d76", document=update_doc)

In [ ]:
vectorstore_from_dir.get(ids= "863c6520-c00d-4e73-b4d6-d94a9de52d76")

deleting documents from a vectorstore

In [ ]:
vectorstore_from_dir.delete(ids=["863c6520-c00d-4e73-b4d6-d94a9de52d76"])

In [ ]:
vectorstore_from_dir.get(ids= "863c6520-c00d-4e73-b4d6-d94a9de52d76")

---
---

## <b>Retrieval</b>

Some algorithms for retrieving documents from a vectorstore
* similarity search
* maximal marginal relevance search

<u> Steps in retrieving a documents </u>
1. Define a question related the documents topic
2. Create a vector representation of the question.
3. Retrieve a pre-selected number of documents relevant to this question.
4. Feed the documents to an LLM to devise a response

In [ ]:
question = "What programming language do data scientist use?"

### <b><u><i> Similarity Search </b></u></i>

In [ ]:
retrieved_docs = vectorstore.similarity_search(query=question, k = 5) # k is the number of similar doc it pull out

In [ ]:
rprint(retrieved_docs)

### <b><u><i>Maximal Marginal Relevance (MMR)

In [ ]:
# question = "What software do data scientists use?"
# retrieved_docs = vectorstore.similarity_search(
#     query=question,
#     k=3
# )
question = "What software do data scientists use?"
retrieved_docs = vectorstore.max_marginal_relevance_search(
    query=question,
    k=3,
    lambda_mult=0.1, # control diversity when retrieving docs
    filter= {"Lecture title": "Programming Languages & Software Employed in Data Science - All the Tools You Need"}
)

In [ ]:
rprint(retrieved_docs)

In [ ]:
len(vectorstore.get()["documents"])

vector-backed re

In [ ]:
retriever = vectorstore.as_retriever(
    search_type= "mmr",
    search_kwargs = {
        "k": 3,
        "lambda_mult": 0.7 
    }
)

In [ ]:
rprint(retriever)

In [ ]:
retrieve_docs = retriever.invoke(question)

In [ ]:
rprint(retrieve_docs)

---
---

## <b> Generation </b>

---
---